#### Portfolio Risk and R&D Drug Pipeline Success with Python Classes
An exploration of using object oriented programing to quantify the following: 
- VaR and risk free rate for general portfolios
- R&D Probability of success, NPV, and probability of negative return using Monte Carlo Simulations

In [4]:
# Let's first understand why quant risk analysts are important as someone considering that area? 
# I see value in mitigating scientific risk ensuring that we achieve Go/No-Go deadlines. 
# My understanding is that quant risk analsts use math and stats to translate complex financial threats into measurable, data-driven decisions. 
# By quantifying potential losses in exact numbers, they protect institutions from failures, ensure regulatory compliance, and optimize capital allocation
# They protect capital and ensure stakeholder communication (which is something I have worked with from a STEM perspective)
# Furthermore, they consider credit, market, and liquidity risks 

import scipy.stats as stats 
import numpy as np

class PortfolioRisk:

    # This part of the class attribute applies to all instances
    confidence_level = 0.99
    risk_free_rate = 0.04

    def __init__(self, portfolio_name, initial_value, volatility): # What are the key defining factors of a portfolio
        self.portfolio_name = portfolio_name
        self.initial_value = initial_value
        self.volatility = volatility 

    def calculate_var(self):
        # Calculates standard historical value at risk (VaR)
        # 99% VaR uses z-score approx to 2.33 

        # old method of calculating
        z_score = 2.33 
        var_amount = self.initial_value*self.volatility

        return var_amount

    def calculate_new_var(self):
    
            # new and correted method of calculating
            z_score = stats.norm.ppf(self.confidence_level)
            t_horizon = 10 # looking at the risk for 10 days
            var_amount_updated = self.initial_value*self.volatility*z_score*np.sqrt(t_horizon)
            return var_amount_updated

# Create instances 
tech_fund = PortfolioRisk("Tech Growth Fund", 1E6, 0.02)
bond_fund = PortfolioRisk("Government Bond Fund", 5E5, 0.005)
# ESG- environment 

# Let's print 
print(f"Risk-Free Rate is: {PortfolioRisk.risk_free_rate*100}%")
print(f"Tech Fund Corrected VaR is: {tech_fund.calculate_new_var():,.2f}")


Risk-Free Rate is: 4.0%
Tech Fund Corrected VaR is: 147,131.16


In [7]:
# Now let's create a class attribute that dictates the number of simulations and an instance method that forcasts the potential future portfolio drawdowns or credit exporesure
# I can also build a borrower class that applies class level constraints and calculates the probability of default based on instnace attributes 
# Resources: 
# https://www.drugpatentwatch.com/blog/a-strategic-investors-guide-to-pharmaceutical-portfolio-risk-assessment/ 
# https://www.oecd.org/en/data/datasets/research-and-development-statistics.html

import numpy as np 

class RDProjectSimulation:

    #Class Attributes: Standard parameters applied to all simulations
    num_simulations = 10000
    discount_rate = 0.15 # 15% hurdle rate for R&D risk

    def __init__(self, name, launch_cost, phase_success_rate, base_revenue, revenue_volatility):
        # Instance Attributes: Unique project variables
        self.name = name
        self.launch_cost = launch_cost
        # I can make one for phase I, phase II, and phase III 
        self.phase_success_rate = phase_success_rate # Probability of surviving R&D
        self.base_revenue = base_revenue
        self.revenue_volatility = revenue_volatility

    def run_monte_carlo(self):
        # Access class attribute using self.num_simulations
        np.random.seed(np.random.randint(30,60)) # Ensures reproducible results

        # Simulate R&D Binary Success/Failure (Binomial distribution)
        # Success (moves to market), 0 = Failure (sunk cost, 0 revenue)
        success_trials = np.random.binomial(1, self.phase_success_rate, self.num_simulations)

        # Simulate Market Revenue Volatility (Lognormal distribution)
        # Revenue cannot be negative, making lognormal ideal for R&D market sizing
        simulated_revenues = np.random.lognormal(
            mean=np.log(self.base_revenue),
            sigma=self.revenue_volatility,
            size=self.num_simulations
        )

        # Calculate NPV for each simulation trial (next time I could vectorize this)
        npv_outcomes = []   

        for i in range(self.num_simulations):
            if success_trials[i] == 1:
            # Project succeeds: Calculate discounted revenue minus launch cost
                discounted_rev = simulated_revenues[i] / (1 + self.discount_rate)
                npv = discounted_rev - self.launch_cost
        
            else:
            # Project fails in R&D: Only loss is the initial R&D/launch cost risk
                npv = -self.launch_cost

            npv_outcomes.append(npv)

        return np.array(npv_outcomes)

    def calculate_var(self, npv_outcomes, confidence_level=0.95):
        cutoff_percentile = (1 - confidence_level)*100
        var_threshold = np.percentile(npv_outcomes, cutoff_percentile)

        return -var_threshold if var_threshold < 0 else 0.0

# Execution and Risk Analysis
# BioTech Drug Candidate: Low success rate (20%), massive upside ($50M base, 50% volatility)
biotech_project = RDProjectSimulation(
name="Antigen Treatment",
launch_cost=5_000_000,
phase_success_rate=0.20,
base_revenue=50_000_000,
revenue_volatility=0.50
)

# Run simulation based on the 10,000 trials class attribute
results = biotech_project.run_monte_carlo()

# Extract key risk metrics
expected_value = np.mean(results)
prob_of_loss = np.mean(results < 0)
var_95 = biotech_project.calculate_var(results, confidence_level=0.95)

print(f"Project: {biotech_project.name}")
print(f"Global Simulation Scale: {RDProjectSimulation.num_simulations:,} trials")
print(f"Expected NPV: ${expected_value:,.2f}")
print(f"Probability of negative return: {prob_of_loss * 100:.1f}%") 
print(f"95% Simulated VaR: ${var_95:,.2f}")

Project: Antigen Treatment
Global Simulation Scale: 10,000 trials
Expected NPV: $5,109,858.72
Probability of negative return: 79.6%
95% Simulated VaR: $5,000,000.00


In [ ]:
# What does this mean potentially? 

# This is a high risk high reward payoff profile. 
# This highlights why diversification matters. 
# Furthermore, this highlights that the project is a go based on risk-adjusted
# return basis, but that is just my understanding. 
# It also means the project faces an 80% catatrophic failure. 